In [9]:
from openpyxl import load_workbook


def write_to_excel(plot_name, product, value, file_path, sheet_name):
    book = load_workbook(file_path)
    sheet = book[sheet_name]

    # 找到对应的行
    row = None
    for idx, row_data in enumerate(sheet.iter_rows(min_row=2, values_only=True)):
        if row_data[1] == plot_name:
            row = idx + 2  # 加2是因为第一行是表头
            break
    if row is None:
        return

    # 找到对应的列
    col = None
    for idx, cell in enumerate(sheet[1]):
        if cell.value == product:
            col = idx  # 列索引
            break
    if col is None:
        return

    # 写入数据
    sheet.cell(row=row, column=col + 1, value=value)  # column+1是因为第一列是季度
    book.save(file_path)

# write_to_excel('A1', '黄豆', 100, "result1_1(2).xlsx", '2024')

In [4]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, LpStatus

# 读取附件中的数据
attachment_1 = pd.read_excel('附件1.xlsx')  # 包含地块信息
attachment_2 = pd.read_excel('附件2.xlsx')  # 包含作物信息和 2023 年的统计数据

plots = attachment_1[['地块名称', '地块类型', '地块面积/亩']].to_dict('records')
crops = attachment_2[['作物名称', '作物类型', '种植面积/亩']].to_dict('records')

seasons = range(1, 3) # 季度 1 - 2

# 定义优化问题
model = LpProblem(name="Crop-Optimization", sense=LpMaximize)

years = range(2024, 2031)  # 从 2024 年到 2030 年

# 定义作物种植的季节性限制
seasonal_limits = {
    '平旱地': {'粮食类作物': 1, '水稻': 0, '蔬菜': 0, '食用菌': 0},
    '梯田': {'粮食类作物': 1, '水稻': 0, '蔬菜': 0, '食用菌': 0},
    '山坡地': {'粮食类作物': 1, '水稻': 0, '蔬菜': 0, '食用菌': 0},
    '水浇地': {'粮食类作物': 0, '水稻': 1, '蔬菜': 2, '食用菌': 0},
    '普通大棚': {'粮食类作物': 0, '水稻': 0, '蔬菜': 2, '食用菌': 1},
    '智慧大棚': {'粮食类作物': 0, '水稻': 0, '蔬菜': 2, '食用菌': 0},
}

# 决策变量：x[i][j][s][t] --> 第 t 年地块 i 在季度 s 种植作物 j 的面积
x = LpVariable.dicts("x",
                     ((i['地块名称'], j['作物名称'], s, t) for i in plots for j in crops for s in seasons for t in years),
                     lowBound=0,
                     cat="Continuous")

# # 决策变量：x[i][j][t] --> 第 t 年地块 i 种植作物 j 的面积
# x = LpVariable.dicts("x",
#                      ((i['地块名称'], j['作物名称'], t) for i in plots for j in crops for t in years),
#                      lowBound=0,
#                      cat="Continuous")

# 辅助变量：z[i][j][t] --> 实际销售的作物产量（不能超过总产量和预期销售量的最小值）
z = LpVariable.dicts("z",
                     ((i['地块名称'], j['作物名称'], t) for i in plots for j in crops for t in years),
                     lowBound=0,
                     cat="Continuous")

# 二进制变量：y[i][j][t] --> 第 t 年地块 i 是否种植作物 j（1 表示种植，0 表示不种植）
y = LpVariable.dicts("y",
                     ((i['地块名称'], j['作物名称'], t) for i in plots for j in crops for t in years),
                     lowBound=0,
                     upBound=1, cat="Binary")

# 目标函数：最大化收益
# model += lpSum(z[i['地块名称'], j['作物名称'], t] * j['种植面积/亩'] - x[i['地块名称'], j['作物名称'], t] * j['种植面积/亩'] for i in plots for j in crops for t in years)
# 目标函数：最大化收益
model += lpSum(z[i['地块名称'], j['作物名称'], t] * j['种植面积/亩'] - x[i['地块名称'], j['作物名称'], t] * j['种植面积/亩'] for i in plots for j in crops for t in years)

# 约束条件
# 1. 每个地块每年的总种植面积不能超过其实际面积
for i in plots:
    for t in years:
        model += lpSum(x[i['地块名称'], j['作物名称'], t] for j in crops) <= i['地块面积/亩']
        
# 2. 作物实际销售产量 z[i][j][t] 不能超过作物的总产量和预期销售量
for i in plots:
    for j in crops:
        for t in years:
            model += z[i['地块名称'], j['作物名称'], t] <= x[i['地块名称'], j['作物名称'], t] * j['种植面积/亩']
            model += z[i['地块名称'], j['作物名称'], t] <= j['种植面积/亩']

# 3. 不重茬约束：同一地块不能连续两年种植相同作物
# 使用二进制变量 y 来表示是否种植
for i in plots:
    for j in crops:
        for t in range(2025, 2031):  # 确保 t 和 t-1 之间没有种植相同作物
            model += y[i['地块名称'], j['作物名称'], t] + y[i['地块名称'], j['作物名称'], t - 1] <= 1
            
# 4. 每三年内必须种植一次豆类作物
for i in plots:
    for t in range(2024, 2028):
        model += lpSum(y[i['地块名称'], j['作物名称'], t + k] for j in
                       crops if j['作物类型'] == '豆类' for k in range(3)) >= 1
        
# 5. 面积约束：如果某地块某年种植某作物，则种植面积必须大于 0
for i in plots:
    for j in crops:
        for t in years:
            model += x[i['地块名称'], j['作物名称'], t] <= y[i['地块名称'], j['作物名称'], t] * i['地块面积/亩']
    
# 6. 季度约束 
# 季度约束：确保每种作物在每个地块的种植次数不超过其允许的最大次数
for i in plots:
    for j in crops:
        for s in seasons:
            for t in years:
                allowed_seasons = seasonal_limits[i['地块类型']].get(j['作物类型'], 0)
                model += x[i['地块名称'], j['作物名称'], s, t] <= allowed_seasons * y[i['地块名称'], j['作物名称'], t]

# 求解模型
status = model.solve()

# 检查求解状态
if status == LpStatus.Optimal:
    print("模型求解成功")
else:
    print("模型求解失败")

# for i in plots:
#     for j in crops:
#         for t in years:
#             if x[i['地块名称'], j['作物名称'], t].value() > 0:
#                 write_to_excel(i['地块名称'], j['作物名称'], x[i['地块名称'], j['作物名称'], t].value(), "result1_1(3).xlsx", str(t))
#                 print(f"地块 {i['地块名称']} 在第 {t} 年种植 {j['作物名称']} {x[i['地块名称'], j['作物名称'], t].value()} 亩")
#         
for i in plots:
    for j in crops:
        for s in seasons:
            for t in years:
                if x[i['地块名称'], j['作物名称'], s, t].value() > 0:
                    print(f"地块 {i['地块名称']} 在第 {t} 年，第 {s} 季度种植 {j['作物名称']} {x[i['地块名称'], j['作物名称'], s, t].value()} 亩")

KeyError: ('A1', '小麦', 2024)

SyntaxError: invalid syntax (3727444082.py, line 56)

In [ ]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, LpStatus

# 读取附件中的数据
attachment_1 = pd.read_excel('附件1.xlsx')  # 包含地块信息
attachment_2 = pd.read_excel('附件2.xlsx')  # 包含作物信息和 2023 年的统计数据

plots = attachment_1[['地块名称', '地块类型', '地块面积/亩']].to_dict('records')
crops = attachment_2[['作物名称', '作物类型', '种植面积/亩']].to_dict('records')

seasons = range(1, 3) # 季度 1 - 2

# 定义优化问题
model = LpProblem(name="Crop-Optimization", sense=LpMaximize)

years = range(2024, 2031)  # 从 2024 年到 2030 年

# 定义作物种植的季节性限制
seasonal_limits = {
    '平旱地': {'粮食类作物': 1, '水稻': 0, '蔬菜': 0, '食用菌': 0},
    '梯田': {'粮食类作物': 1, '水稻': 0, '蔬菜': 0, '食用菌': 0},
    '山坡地': {'粮食类作物': 1, '水稻': 0, '蔬菜': 0, '食用菌': 0},
    '水浇地': {'粮食类作物': 0, '水稻': 1, '蔬菜': 2, '食用菌': 0},
    '普通大棚': {'粮食类作物': 0, '水稻': 0, '蔬菜': 2, '食用菌': 1},
    '智慧大棚': {'粮食类作物': 0, '水稻': 0, '蔬菜': 2, '食用菌': 0},
}

# 决策变量：x[i][j][s][t] --> 第 t 年地块 i 在季度 s 种植作物 j 的面积
x = LpVariable.dicts("x",
                     ((i['地块名称'], j['作物名称'], s, t) for i in plots for j in crops for s in seasons for t in years),
                     lowBound=0,
                     cat="Continuous")

# # 决策变量：x[i][j][t] --> 第 t 年地块 i 种植作物 j 的面积
# x = LpVariable.dicts("x",
#                      ((i['地块名称'], j['作物名称'], t) for i in plots for j in crops for t in years),
#                      lowBound=0,
#                      cat="Continuous")

# 辅助变量：z[i][j][t] --> 实际销售的作物产量（不能超过总产量和预期销售量的最小值）
z = LpVariable.dicts("z",
                     ((i['地块名称'], j['作物名称'], t) for i in plots for j in crops for t in years),
                     lowBound=0,
                     cat="Continuous")

# 二进制变量：y[i][j][t] --> 第 t 年地块 i 是否种植作物 j（1 表示种植，0 表示不种植）
y = LpVariable.dicts("y",
                     ((i['地块名称'], j['作物名称'], t) for i in plots for j in crops for t in years),
                     lowBound=0,
                     upBound=1, cat="Binary")

# 目标函数：最大化收益
# model += lpSum(z[i['地块名称'], j['作物名称'], t] * j['种植面积/亩'] - x[i['地块名称'], j['作物名称'], t] * j['种植面积/亩'] for i in plots for j in crops for t in years)

# 约束条件
# 1. 每个地块每年的总种植面积不能超过其实际面积
for i in plots:
    for t in years:
        model += lpSum(x[i['地块名称'], j['作物名称'], t] for j in crops) <= i['地块面积/亩']

# 2. 作物实际销售产量 z[i][j][t] 不能超过作物的总产量和预期销售量
for i in plots:
    for j in crops:
        for t in years:
            model += z[i['地块名称'], j['作物名称'], t] <= x[i['地块名称'], j['作物名称'], t] * j['种植面积/亩']
            model += z[i['地块名称'], j['作物名称'], t] <= j['种植面积/亩']

# 3. 不重茬约束：同一地块不能连续两年种植相同作物
# 使用二进制变量 y 来表示是否种植
for i in plots:
    for j in crops:
        for t in range(2025, 2031):  # 确保 t 和 t-1 之间没有种植相同作物
            model += y[i['地块名称'], j['作物名称'], t] + y[i['地块名称'], j['作物名称'], t - 1] <= 1

# 4. 每三年内必须种植一次豆类作物
for i in plots:
    for t in range(2024, 2028):
        model += lpSum(y[i['地块名称'], j['作物名称'], t + k] for j in
                       crops if j['作物类型'] == '豆类' for k in range(3)) >= 1

# 5. 面积约束：如果某地块某年种植某作物，则种植面积必须大于 0
for i in plots:
    for j in crops:
        for t in years:
            model += x[i['地块名称'], j['作物名称'], t] <= y[i['地块名称'], j['作物名称'], t] * i['地块面积/亩']

# 6. 季度约束 
# 季度约束：确保每种作物在每个地块的种植次数不超过其允许的最大次数
for i in plots:
    for j in crops:
        for s in seasons:
            for t in years:
                allowed_seasons = seasonal_limits[i['地块类型']].get(j['作物类型'], 0)
                model += x[i['地块名称'], j['作物名称'], s, t] <= allowed_seasons * y[i['地块名称'], j['作物名称'], t]

# 求解模型
status = model.solve()

# 检查求解状态
if status == LpStatus.Optimal:
    print("模型求解成功")
else:
    print("模型求解失败")

for i in plots:
    for j in crops:
        for s in seasons:
            for t in years:
                if x[i['地块名称'], j['作物名称'], s, t].value() > 0:
                    print(f"地块 {i['地块名称']} 在第 {t} 年，第 {s} 季度种植 {j['作物名称']} {x[i['地块名称'], j['作物名称'], s, t].value()} 亩")

In [ ]:
import pulp
import pandas as pd

# 定义数据结构
crops = list(range(1, 42))
lands = [f"{i}{j}" for i in 'ABCDEFG' for j in range(1, 17)]
years = list(range(1, 8))
seasons = list(range(1, 3))

# 读取数据
production_q1 = pd.read_excel('final_output_1.xlsx', sheet_name='final_output_1', usecols="B", skiprows=1, header=None).values
production_q2 = pd.read_excel('final_output_1.xlsx', sheet_name='final_output_1', usecols="C", skiprows=1, header=None).values
price = pd.read_excel('final_output_1.xlsx', sheet_name='final_output_1', usecols="D", skiprows=1, header=None).values
cost = pd.read_excel('final_output_1.xlsx', sheet_name='final_output_1', usecols="E", skiprows=1, header=None).values
land_area = pd.read_excel('附件1.xlsx', sheet_name='乡村的现有耕地', usecols="C", skiprows=1, header=None).values
# ... 其他数据的读取 ...

# 创建问题实例
model = pulp.LpProblem("Maximize_Profit", pulp.LpMaximize)

# 创建决策变量
x = pulp.LpVariable.dicts("x", (lands, crops, years, seasons), lowBound=0)
excess_half_price = pulp.LpVariable.dicts("excess_half_price", (crops, years, seasons), lowBound=0)

# 定义目标函数
model += pulp.lpSum([price[j-1] * (w_q1 if w_q1 <= production_q1[j-1] else production_q1[j-1]) +
                     price[j-1] * (w_q2 if w_q2 <= production_q2[j-1] else production_q2[j-1]) +
                     0.5 * price[j-1] * excess_half_price[j-1][t-1][s-1] -
                     cost[j-1] * x[(i, j, t, s)]
                     for i in lands for j in crops for t in years for s in seasons])

# 添加约束
for i in lands:
    for t in years:
        for s in seasons:
            model += pulp.lpSum([x[(i, j, t, s)] for j in crops]) <= land_area[i-1]

# ... 其他约束 ...

# 求解问题
model.solve()

# 输出结果
for v in model.variables():
    print(v.name, "=", v.varValue)

# 检查状态
if model.status == pulp.LpStatusOptimal:
    print("Found an optimal solution.")
else:
    print("Optimal solution not found.")